# Demos: 13 Data Formats and Tools (optional)

Each cell below is one demo from the slides; the heading matches the pill on the slide. Run the first cell, then run the demos in any order with Shift+Enter. The same code is in the `.py` files in this folder.

In [ ]:
# Run this cell first: the demos read their data files from the working folder.
import os
from pathlib import Path

COURSE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir())
os.chdir(COURSE / "datasets/demos/13-data-formats-and-tools")
print("Working folder:", Path.cwd())

## demo01 · Typed ingestion

In [ ]:
import pandas as pd

orders = pd.read_csv(
    "orders.csv",
    dtype={"order_id": str, "units": "Int64", "region": "category", "status": "category"},
    parse_dates=["ordered_at"],
    date_format="%Y-%m-%d",
)
print(orders.dtypes)
print(orders.head(4))

inferred = pd.read_csv("orders.csv")
print(inferred["order_id"].head(3).tolist(), inferred["units"].dtype)

## demo02 · Arrow dtypes

In [ ]:
import pandas as pd

arrow = pd.read_csv("orders.csv", dtype={"order_id": "string[pyarrow]"}, dtype_backend="pyarrow")
print(arrow.dtypes)
print(arrow["units"].isna().sum())

## demo03 · pandas answer

In [ ]:
import pandas as pd

orders = pd.read_csv(
    "orders.csv",
    dtype={"order_id": str, "units": "Int64", "region": "category", "status": "category"},
    parse_dates=["ordered_at"],
    date_format="%Y-%m-%d",
)
june_shipped = (
    orders["ordered_at"].ge("2026-06-01")
    & orders["ordered_at"].lt("2026-07-01")
    & orders["status"].eq("shipped")
)
selected = orders.loc[june_shipped & orders["region"].notna() & orders["units"].notna()]
print(june_shipped.sum(), len(selected))

answer = (
    selected.groupby("region", observed=True)
    .agg(units=("units", "sum"), orders=("units", "size"))
    .reset_index()
)
print(answer)

## demo04 · Parquet

In [ ]:
import pandas as pd
import pyarrow.parquet as pq

orders = pd.read_csv(
    "orders.csv",
    dtype={"order_id": str, "units": "Int64", "region": "category", "status": "category"},
    parse_dates=["ordered_at"],
    date_format="%Y-%m-%d",
)

orders.to_parquet("orders.parquet", index=False)
restored = pd.read_parquet("orders.parquet")
print(restored.dtypes)
print(restored["order_id"].head(3).tolist())

print(pq.read_schema("orders.parquet"))
subset = pd.read_parquet("orders.parquet", columns=["region", "units"])
print(subset.columns.tolist())

## demo05 · DuckDB

In [ ]:
import duckdb
import pandas as pd

orders = pd.read_csv(
    "orders.csv",
    dtype={"order_id": str, "units": "Int64", "region": "category", "status": "category"},
    parse_dates=["ordered_at"],
    date_format="%Y-%m-%d",
)

orders.to_parquet("orders.parquet", index=False)
SQL = """
SELECT region, CAST(SUM(units) AS INTEGER) AS units, COUNT(*) AS orders
FROM {source}
WHERE ordered_at >= DATE '2026-06-01' AND ordered_at < DATE '2026-07-01'
  AND status = 'shipped' AND region IS NOT NULL AND units IS NOT NULL
GROUP BY region ORDER BY region
"""
con = duckdb.connect()
con.register("orders", orders)
print(con.execute(SQL.format(source="orders")).df())
print(con.execute(SQL.format(source="read_parquet('orders.parquet')")).df())

plan = con.execute("EXPLAIN " + SQL.format(source="read_parquet('orders.parquet')")).fetchone()[1]
print(plan)
con.close()

## demo06 · Polars basics

In [ ]:
import polars as pl

orders = pl.read_csv("orders.csv", schema_overrides={"order_id": pl.String}, try_parse_dates=True)
print(orders.head(3))
result = (
    orders.filter(pl.col("status") == "shipped")
    .filter(pl.col("region").is_in(["North", "South"]))
    .with_columns(pl.col("units").cast(pl.Float64).alias("units_f"))
    .select("order_id", "region", "units_f")
)
print(result.head(4))

## demo07 · Polars lazy

In [ ]:
from datetime import date

import pandas as pd
import polars as pl

pd.read_csv(
    "orders.csv", dtype={"order_id": str, "units": "Int64"}, parse_dates=["ordered_at"]
).to_parquet("orders.parquet", index=False)

query = (
    pl.scan_parquet("orders.parquet")
    .filter(
        (pl.col("ordered_at") >= pl.lit(date(2026, 6, 1)))
        & (pl.col("ordered_at") < pl.lit(date(2026, 7, 1)))
        & (pl.col("status") == "shipped")
        & pl.col("region").is_not_null()
        & pl.col("units").is_not_null()
    )
    .group_by("region")
    .agg(pl.col("units").sum(), pl.len().alias("orders"))
    .sort("region")
)
print(query.explain())
print(query.collect())

## demo08 · Numba (optional)

In [ ]:
from timeit import timeit

import numpy as np
from numba import njit


def sum_squares(values):
    total = 0.0
    for value in values:
        total += value * value
    return total


compiled = njit(sum_squares)
values = np.random.default_rng(0).random(1_000_000)
compiled(values)  # the first call compiles the function

print(round(sum_squares(values), 3), round(compiled(values), 3), round(np.sum(values**2), 3))
for label, call in [
    ("Python loop", lambda: sum_squares(values)),
    ("Numba", lambda: compiled(values)),
    ("NumPy", lambda: np.sum(values**2)),
]:
    print(f"{label}: {timeit(call, number=5) / 5 * 1000:.2f} ms")